# Imports

In [2]:
import os
import torch
import numpy as np
import open3d as o3d
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
from scene.dataset_readers import sceneLoadTypeCallbacks
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False

def prepare_output_and_logger(args):    
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])
        
    # Set up output folder
    print("Output folder: {}".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    # Create Tensorboard writer
    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer



def training_report(tb_writer, iteration, Ll1, loss, l1_loss, elapsed, testing_iterations, scene : Scene, renderFunc, renderArgs, train_test_exp):
    if tb_writer:
        tb_writer.add_scalar('train_loss_patches/l1_loss', Ll1.item(), iteration)
        tb_writer.add_scalar('train_loss_patches/total_loss', loss.item(), iteration)
        tb_writer.add_scalar('iter_time', elapsed, iteration)

    # Report test and samples of training set
    if iteration in testing_iterations:
        torch.cuda.empty_cache()
        validation_configs = ({'name': 'test', 'cameras' : scene.getTestCameras()}, 
                              {'name': 'train', 'cameras' : [scene.getTrainCameras()[idx % len(scene.getTrainCameras())] for idx in range(5, 30, 5)]})

        for config in validation_configs:
            if config['cameras'] and len(config['cameras']) > 0:
                l1_test = 0.0
                psnr_test = 0.0
                for idx, viewpoint in enumerate(config['cameras']):
                    image = torch.clamp(renderFunc(viewpoint, scene.gaussians, *renderArgs)["render"], 0.0, 1.0)
                    gt_image = torch.clamp(viewpoint.original_image.to("cuda"), 0.0, 1.0)
                    if train_test_exp:
                        image = image[..., image.shape[-1] // 2:]
                        gt_image = gt_image[..., gt_image.shape[-1] // 2:]
                    if tb_writer and (idx < 5):
                        tb_writer.add_images(config['name'] + "_view_{}/render".format(viewpoint.image_name), image[None], global_step=iteration)
                        if iteration == testing_iterations[0]:
                            tb_writer.add_images(config['name'] + "_view_{}/ground_truth".format(viewpoint.image_name), gt_image[None], global_step=iteration)
                    l1_test += l1_loss(image, gt_image).mean().double()
                    psnr_test += psnr(image, gt_image).mean().double()
                psnr_test /= len(config['cameras'])
                l1_test /= len(config['cameras'])          
                print("\n[ITER {}] Evaluating {}: L1 {} PSNR {}".format(iteration, config['name'], l1_test, psnr_test))
                if tb_writer:
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - l1_loss', l1_test, iteration)
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - psnr', psnr_test, iteration)

        if tb_writer:
            tb_writer.add_histogram("scene/opacity_histogram", scene.gaussians.get_opacity, iteration)
            tb_writer.add_scalar('total_points', scene.gaussians.get_xyz.shape[0], iteration)
        torch.cuda.empty_cache()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Parser Setting

In [3]:
parser = ArgumentParser(description="Training script parameters")
lp = ModelParams(parser)
op = OptimizationParams(parser)
pp = PipelineParams(parser)
parser.add_argument('--ip', type=str, default="127.0.0.1")
parser.add_argument('--port', type=int, default=6009)
parser.add_argument('--debug_from', type=int, default=-1)
parser.add_argument('--detect_anomaly', action='store_true', default=False)
parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--quiet", action="store_true")
parser.add_argument('--disable_viewer', action='store_true', default=False)
parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
parser.add_argument("--start_checkpoint", type=str, default=None)

# In Jupyter, parse_known_args avoids runtime arguments such as -f that are injected by the notebook kernel.
args, _ = parser.parse_known_args([])
args.save_iterations.append(args.iterations)

# Convert the parser namespace into the lightweight argument objects used by the training code.
model_args = lp.extract(args)
opt_args = op.extract(args)
pipe_args = pp.extract(args)
model_args.source_path = os.path.join(model_args.source_path,"GaussianTest/Test2") 
# source_path is hardcoded on purpose for this tutorial, but you can change it to your own dataset path.

print("Loaded parser-backed arguments")
print("sh_degree:", model_args.sh_degree)
print("optimizer_type:", opt_args.optimizer_type)
print("source path: ", model_args.source_path)

Loaded parser-backed arguments
sh_degree: 3
optimizer_type: default
source path:  c:\Dev\gaussian-splatting-for-practice\GaussianTest/Test2


# Setup Before Training

In [4]:
first_iter = 0
tb_writer = prepare_output_and_logger(model_args)
gaussians = GaussianModel(model_args.sh_degree, opt_args.optimizer_type)
scene = Scene(model_args, gaussians)
gaussians.training_setup(opt_args)
if args.checkpoint_iterations:
    (model_params, first_iter) = torch.load(args.checkpoint)
    gaussians.restore(model_params, opt_args)

bg_color = [1, 1, 1] if model_args.white_background else [0, 0, 0]
background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

iter_start = torch.cuda.Event(enable_timing = True)
iter_end = torch.cuda.Event(enable_timing = True)

use_sparse_adam = opt_args.optimizer_type == "sparse_adam" and SPARSE_ADAM_AVAILABLE 
depth_l1_weight = get_expon_lr_func(opt_args.depth_l1_weight_init, opt_args.depth_l1_weight_final, max_steps=opt_args.iterations)

viewpoint_stack = scene.getTrainCameras().copy()
viewpoint_indices = list(range(len(viewpoint_stack)))
ema_loss_for_log = 0.0
ema_Ll1depth_for_log = 0.0

progress_bar = tqdm(range(first_iter, opt_args.iterations), desc="Training progress")
first_iter += 1

Output folder: ./output/e7483cd0-2
Tensorboard not available: not logging progress
Reading camera 25/25
Loading Training Cameras


c:\Users\COM\anaconda3\envs\gaussian_splatting\lib\site-packages\torch\cuda\__init__.py:218: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Loading Test Cameras
Number of points at initialisation :  1768


Training progress:   0%|          | 0/30000 [00:00<?, ?it/s]

# Start Training

In [5]:
for iteration in range(first_iter, 1001):
    iter_start.record()
    
    gaussians.update_learning_rate(iteration)
    
    # Every 1000 its we increase the levels of SH up to a maximum degree
    if iteration % 1000 == 0:
        gaussians.oneupSHdegree()
    
    # Pick a random Camera
    if not viewpoint_stack:
        viewpoint_stack = scene.getTrainCameras().copy()
        viewpoint_indices = list(range(len(viewpoint_stack)))
    rand_idx = randint(0, len(viewpoint_indices) - 1)
    viewpoint_cam = viewpoint_stack.pop(rand_idx)
    vind = viewpoint_indices.pop(rand_idx)
    
    # Render
    if (iteration - 1) == args.debug_from:
        pipe_args.debug = True
    
    bg = torch.rand((3), device="cuda") if opt_args.random_background else background
    
    render_pkg = render(viewpoint_cam, gaussians, pipe_args, bg, use_trained_exp=model_args.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)
    image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]
    
    if viewpoint_cam.alpha_mask is not None:
        alpha_mask = viewpoint_cam.alpha_mask.cuda()
        image *= alpha_mask
    
    # Loss
    gt_image = viewpoint_cam.original_image.cuda()
    Ll1 = l1_loss(image, gt_image)
    if FUSED_SSIM_AVAILABLE:
        ssim_value = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))
    else:
        ssim_value = ssim(image, gt_image)
    
    loss = (1.0 - opt_args.lambda_dssim) * Ll1 + opt_args.lambda_dssim * (1.0 - ssim_value)
    
    # Depth regularization
    Ll1depth_pure = 0.0
    if depth_l1_weight(iteration) > 0 and viewpoint_cam.depth_reliable:
        invDepth = render_pkg["depth"]
        mono_invdepth = viewpoint_cam.invdepthmap.cuda()
        depth_mask = viewpoint_cam.depth_mask.cuda()
    
        Ll1depth_pure = torch.abs((invDepth  - mono_invdepth) * depth_mask).mean()
        Ll1depth = depth_l1_weight(iteration) * Ll1depth_pure 
        loss += Ll1depth
        Ll1depth = Ll1depth.item()
    else:
        Ll1depth = 0
    
    loss.backward()
    
    iter_end.record()
    
    with torch.no_grad():
        # Progress bar
        ema_loss_for_log = 0.4 * loss.item() + 0.6 * ema_loss_for_log
        ema_Ll1depth_for_log = 0.4 * Ll1depth + 0.6 * ema_Ll1depth_for_log
    
        if iteration % 10 == 0:
            progress_bar.set_postfix({"Loss": f"{ema_loss_for_log:.{7}f}", "Depth Loss": f"{ema_Ll1depth_for_log:.{7}f}"})
            progress_bar.update(10)
        if iteration == opt_args.iterations:
            progress_bar.close()
    
        # Log and save
        training_report(tb_writer, iteration, Ll1, loss, l1_loss, iter_start.elapsed_time(iter_end), args.test_iterations, scene, render, (pipe_args, background, 1., SPARSE_ADAM_AVAILABLE, None, model_args.train_test_exp), model_args.train_test_exp)
        if (iteration in args.save_iterations):
            print("\n[ITER {}] Saving Gaussians".format(iteration))
            scene.save(iteration)
    
        # Densification
        if iteration < opt_args.densify_until_iter:
            # Keep track of max radii in image-space for pruning
            gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
            gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)
    
            if iteration > opt_args.densify_from_iter and iteration % opt_args.densification_interval == 0:
                size_threshold = 20 if iteration > opt_args.opacity_reset_interval else None
                gaussians.densify_and_prune(opt_args.densify_grad_threshold, 0.005, scene.cameras_extent, size_threshold, radii)
    
            if iteration % opt_args.opacity_reset_interval == 0 or (model_args.white_background and iteration == opt_args.densify_from_iter):
                gaussians.reset_opacity()

Training progress:   2%|▏         | 600/30000 [00:15<09:33, 51.29it/s, Loss=0.3039724, Depth Loss=0.0000000] 

--- 증식 전 총 가우시안: 1768 ---
 복제(Clone)된 가우시안 개수: 13
Clone 직후 총 가우시안: 1781
 분할(Split)된 원본 가우시안 개수: 661 (이것이 2배로 쪼개짐)
Split 직후 총 가우시안: 2442
Prune 직후 최종 가우시안: 2442


Training progress:   2%|▏         | 710/30000 [00:17<08:27, 57.68it/s, Loss=0.3290529, Depth Loss=0.0000000]

--- 증식 전 총 가우시안: 2442 ---
 복제(Clone)된 가우시안 개수: 51
Clone 직후 총 가우시안: 2493
 분할(Split)된 원본 가우시안 개수: 973 (이것이 2배로 쪼개짐)
Split 직후 총 가우시안: 3466
Prune 직후 최종 가우시안: 3466


Training progress:   3%|▎         | 810/30000 [00:19<07:50, 62.03it/s, Loss=0.3407160, Depth Loss=0.0000000]

--- 증식 전 총 가우시안: 3466 ---
 복제(Clone)된 가우시안 개수: 143
Clone 직후 총 가우시안: 3609
 분할(Split)된 원본 가우시안 개수: 1176 (이것이 2배로 쪼개짐)
Split 직후 총 가우시안: 4785
Prune 직후 최종 가우시안: 4785


Training progress:   3%|▎         | 910/30000 [00:20<06:31, 74.34it/s, Loss=0.4343358, Depth Loss=0.0000000]

--- 증식 전 총 가우시안: 4785 ---
 복제(Clone)된 가우시안 개수: 395
Clone 직후 총 가우시안: 5180
 분할(Split)된 원본 가우시안 개수: 1504 (이것이 2배로 쪼개짐)
Split 직후 총 가우시안: 6684
Prune 직후 최종 가우시안: 6684


Training progress:   3%|▎         | 1000/30000 [00:21<05:32, 87.12it/s, Loss=0.3979448, Depth Loss=0.0000000]

--- 증식 전 총 가우시안: 6684 ---
 복제(Clone)된 가우시안 개수: 905
Clone 직후 총 가우시안: 7589
 분할(Split)된 원본 가우시안 개수: 1850 (이것이 2배로 쪼개짐)
Split 직후 총 가우시안: 9439
Prune 직후 최종 가우시안: 9439


In [6]:
print(f"iteration: {iteration}")
print(f"opt_args.densify_from_iter: {opt_args.densify_from_iter}")
print(f"opt_args.densification_interval: {opt_args.densification_interval}")
print(f"iteration % opt_args.densification_interval: {iteration % opt_args.densification_interval}")
print(f"opt_args.opacity_reset_interval: {opt_args.opacity_reset_interval}")
print(f"size_threshold: {size_threshold}")
print(f"viewspace_point_tensor.shape: {viewspace_point_tensor.shape}")
print(f"viewspace_point_tensor: {viewspace_point_tensor.any()}")


iteration: 1000
opt_args.densify_from_iter: 500
opt_args.densification_interval: 100
iteration % opt_args.densification_interval: 0
opt_args.opacity_reset_interval: 3000
size_threshold: None
viewspace_point_tensor.shape: torch.Size([6684, 3])
viewspace_point_tensor: False


In [8]:
print(f"opt_args.densify_grad_threshold: {opt_args.densify_grad_threshold}")
print(f"scene.cameras_extent:            {scene.cameras_extent}")
print(f"gaussians.get_scaling:           {gaussians.get_scaling}")
print(f"gaussians.percent_dense:         {gaussians.percent_dense}")

opt_args.densify_grad_threshold: 0.0002
scene.cameras_extent:            5.079339504241943
gaussians.get_scaling:           tensor([[0.0712, 0.0712, 0.0712],
        [0.0731, 0.0731, 0.0731],
        [0.0732, 0.0732, 0.0732],
        ...,
        [0.0620, 0.0620, 0.0620],
        [0.0321, 0.0321, 0.0321],
        [0.2691, 0.2691, 0.2691]], device='cuda:0', grad_fn=<ExpBackward0>)
gaussians.percent_dense:         0.01


In [9]:
a=gaussians.percent_dense
b=scene.cameras_extent
print(a*b)

0.050793395042419434
